[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P14/P14-1_Interpoler.ipynb)

# Interpoler un abaque

Chapitre 14 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre dossier de calcul cite un coefficient de concentration de contrainte lu sur un abaque : huit points relevés, et le trou de votre pièce tombe entre deux d'entre eux. Un module d'Young n'est donné qu'à quelques températures, et votre pièce chauffe à une autre. Une table a toujours des lacunes entre ses lignes ; vous voudriez que l'ordinateur lise entre les points, chiffre de combien il peut se tromper, et refuse de répondre hors de la table.

**Objectif terminal.** Ce chapitre montre comment, avec le module `scipy.interpolate` de SciPy.

**Objectif instrumental.**

- Lire une valeur entre deux points d'un tableau avec `np.interp`, et en éviter les pièges.
- Lisser un abaque par une spline cubique, lire sa pente et reconnaître quand elle oscille.
- Lire un abaque à deux entrées, sur une grille ou sur des points dispersés.
- Décider ce que l'on fait hors du domaine de l'abaque : refuser, prolonger ou marquer.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 3 et 6, après une filière scientifique.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 14.1 | Lire Kt entre deux points d'abaque | 10 min |
| 14.2 | Kt par spline, et sa pente | 15 min |
| 14.3 | Abaque à deux entrées, et extrapolation refusée (défi) | 25 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("scipy", version("scipy"))

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Fonctions du module du livre (`meca.py`)

Le code ci-dessous reprend, tel qu'il est écrit dans le livre, les fonctions du module employées dans ce notebook ; ce code est fourni tel quel.

In [ ]:
def kt_trou(d_mm, W_mm):
    """Kt d'une plaque de largeur W percée d'un trou central de diamètre d, en traction.

    Formule de Heywood : Kt = 2 + (1 - d/W)**3, rapportée à la contrainte nette F / ((W - d) e).
    Source : R. B. Heywood, Designing by Photoelasticity, Chapman and Hall, 1952 ; formule reprise
    par W. D. Pilkey et D. F. Pilkey, Peterson's Stress Concentration Factors, Wiley, et par
    Roark's Formulas for Stress and Strain, McGraw-Hill. Domaine : 0 < d/W < 1.

    >>> round(kt_trou(10.0, 50.0), 3)
    2.512
    >>> kt_trou(60.0, 50.0) != kt_trou(60.0, 50.0)     # nan hors du domaine
    True
    """
    rapport = d_mm / W_mm
    if rapport <= 0 or rapport >= 1:
        return float("nan")      # hors du domaine : pas de trou, ou plus de matière
    return 2 + (1 - rapport)**3

### Préparation

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline, RegularGridInterpolator
W_mm = 50.0                                              # largeur de la plaque, comme au chapitre 3

### Section 14.1 : un abaque se lit entre deux points.

In [ ]:
# notion: interp
temperatures_C = [20.0, 100.0, 200.0, 300.0]             # module d'Young d'un acier, relevé en quatre températures
modules_GPa = [210.0, 205.0, 198.0, 190.0]
module_150 = np.interp(150.0, temperatures_C, modules_GPa)     # entre 100 et 200 : milieu des deux valeurs
print(f"module d'Young à 150 degrés : {module_150:.1f} GPa")
assert abs(module_150 - 201.5) < 1e-12, "milieu de 205 et 198"
assert np.interp(20.0, temperatures_C, modules_GPa) == 210.0, "aux nœuds, la valeur du tableau"
assert (
    np.interp(0.0, temperatures_C, modules_GPa) == 210.0
), "hors du tableau, np.interp garde la valeur de l'extrémité"
# L'auteur place une annotation sur une courbe avec np.interp : l'ordonnée de la courbe à l'abscisse du texte.
x_courbe = np.linspace(0.0, 1.0, 11)
y_courbe = x_courbe**2
y_texte = np.interp(0.45, x_courbe, y_courbe)
assert abs(y_texte - 0.205) < 1e-12 and y_texte > 0.45**2, "la corde est au-dessus de la parabole"

## Exercice 14.1 : Lire Kt entre deux points d'abaque (10 min)

Avec l'abaque de huit points ($W = 50$ mm, $d/W$ de $0{,}1$ à $0{,}8$), lisez $K_t$ pour $d/W = 0{,}35$ avec `np.interp`. Comparez à la formule de Heywood, et exprimez l'écart en pourcentage de la formule.

**Résultat attendu.** $K_t$ lu de 2,2795 ; formule de 2,2746 ; lecture trop forte de 0,2 %.

**Seconde voie.** La formule de Heywood du chapitre 3, avec $d = 17{,}5$ mm et $W = 50$ mm, donne 2,2746.

**Indication.** Le rapport $0{,}35$ tombe entre les nœuds $0{,}3$ et $0{,}4$ : `np.interp` prend le milieu de leurs deux valeurs. L'écart relatif est `100 * (lu - exact) / exact`. Il est positif : la corde passe au-dessus d'une courbe convexe.

Les contrôles de la cellule lisent les variables `ecart_relatif`, `kt_exact`, `kt_lu`, `kt_noeuds` : gardez ces noms.

In [ ]:
# Votre code :
assert abs(kt_lu - 2.2795) < 5e-5, "Kt lu attendu : 2,2795"
assert abs(ecart_relatif - 0.2) < 0.05, "surestimation attendue : 0,2 %"
assert kt_lu > kt_exact, "la fonction est convexe : la corde passe au-dessus"
assert kt_noeuds[2] > kt_lu > kt_noeuds[3], "entre les nœuds d/W = 0,3 et 0,4"
print("Exercice 14.1 : contrôles passés")

### Section 14.2 : une spline lisse l'abaque.

In [ ]:
# notion: spline
spline = CubicSpline(rapports, kt_noeuds)                # une cubique par intervalle, raccordées en pente et en courbure
kt_spline = float(spline(0.35))
pente_spline = float(spline(0.35, 1))                    # le deuxième argument demande la dérivée première
pente_exacte = -3 * (1 - 0.35)**2                         # dérivée de 2 + (1 - r)^3
naturelle = CubicSpline(rapports, kt_noeuds, bc_type="natural")
ecart_naturelle = abs(float(naturelle(0.35)) - kt_exact)
print(f"spline : Kt {kt_spline:.4f} ; pente {pente_spline:.4f} contre {pente_exacte:.4f} ; extrémités naturelles : écart {ecart_naturelle:.5f}")
assert np.allclose(spline(rapports), kt_noeuds), "la spline passe par tous les nœuds"
assert abs(pente_spline - pente_exacte) < 1e-9, "la pente de la spline est la pente de la formule"
assert ecart_naturelle > 1e-4, "des extrémités à courbure nulle ne conviennent pas à cette courbe"

## Exercice 14.2 : Kt par spline, et sa pente (15 min)

Avec les huit points de l'abaque, lisez $K_t$ à $d/W = 0{,}35$ par `CubicSpline`. Calculez l'écart entre la lecture linéaire de l'exercice précédent et la spline, puis la pente de la spline en ce point. Comparez à la dérivée de la formule par rapport à $d/W$, $-3\,(1 - d/W)^2$.

**Résultat attendu.** Spline de 2,2746, égale à la formule ; écart de la lecture linéaire de 0,0049 ; pente de $-1{,}2675$, égale à la dérivée de la formule.

**Seconde voie.** La formule de Heywood donne 2,2746, et sa dérivée en $0{,}35$ vaut $-3\,(1 - 0{,}35)^2$.

**Indication.** Avec `spline = CubicSpline(rapports, kt_abaque)`, `spline(0.35)` donne la valeur, `spline(0.35, 1)` la pente. La formule de Heywood est un polynôme du troisième degré en $d/W$ : une spline cubique par ses points la reproduit exactement, ce qui n'est pas le cas pour une courbe quelconque. Avec `bc_type="natural"`, la courbure est forcée à zéro aux extrémités, et l'écart devient mesurable (0,0002).

Les contrôles de la cellule lisent la variable `ecart_lineaire` : gardez ce nom.

In [ ]:
# Votre code :
assert abs(kt_spline - 2.2746) < 5e-5, "Kt par spline attendu : 2,2746"
assert (
    abs(kt_spline - kt_exact) < 1e-12
), "la formule est un polynôme du troisième degré : la spline est exacte"
assert abs(ecart_lineaire - 0.0049) < 5e-5, "écart attendu : 0,0049"
print("Exercice 14.2 : contrôles passés")

### Section 14.3 : un abaque à deux entrées demande une grille.

In [ ]:
# notion: grille-interp
# Plaque percée tirée dans deux directions, avec lambda le rapport de la seconde contrainte à la première : le
# coefficient est celui de la formule de Heywood multiplié par (3 - lambda) / 3, qui vaut 1 en traction simple
# (lambda = 0) et 2/3 en traction égale dans les deux directions (lambda = 1), où la plaque infinie donne Kt = 2.
lambdas = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
table = np.array([[kt_trou(r * W_mm, W_mm) * (3 - lam) / 3 for lam in lambdas] for r in rapports])
print("table de l'abaque :", table.shape, "(8 valeurs de d/W, 5 valeurs de lambda)")
assert table.shape == (8, 5), "une ligne par d/W, une colonne par lambda"
assert np.allclose(table[:, 0], kt_noeuds), "pour lambda = 0, la formule de Heywood"
lecture = RegularGridInterpolator((rapports, lambdas), table)           # interpolation linéaire par défaut
kt_2d = float(lecture([[0.35, 0.5]])[0])
kt_2d_exact = kt_trou(0.35 * W_mm, W_mm) * (3 - 0.5) / 3
cubique = RegularGridInterpolator((rapports, lambdas), table, method="cubic")
kt_2d_cubique = float(cubique([[0.35, 0.5]])[0])
print(f"Kt(d/W = 0.35, lambda = 0.5) : grille {kt_2d:.4f} ; grille cubique {kt_2d_cubique:.4f} ; formule {kt_2d_exact:.4f}")
assert abs(lecture([[0.3, 0.25]])[0] - table[2, 1]) < 1e-12, "aux nœuds, la valeur de la table"

In [ ]:
assert kt_2d > kt_2d_exact, "linéaire en d/W : la corde est au-dessus de la courbe"
assert abs(kt_2d_cubique - kt_2d_exact) < 5e-5, "cubique : écart sous 5e-5"

# notion: extrapolation
# Hors de la grille, RegularGridInterpolator refuse par défaut ; np.interp garde la dernière valeur ; prolonger la
# pente des deux derniers nœuds donne un nombre, jamais une garantie. Une fonction qui refuse protège le calcul.
try:
    lecture([[0.9, 0.5]])
    refus = "aucune erreur"
except ValueError:
    refus = "ValueError : d/W = 0,9 est hors de l'abaque"

prolonge = RegularGridInterpolator((rapports, lambdas), table, bounds_error=False, fill_value=None)
kt_prolonge = float(prolonge([[0.9, 0.0]])[0])
kt_borne = float(np.interp(0.9, rapports, kt_noeuds))
kt_090 = kt_trou(0.9 * W_mm, W_mm)

In [ ]:
def kt_abaque(r, lam):
    """Kt de l'abaque à deux entrées, ou ValueError hors du domaine relevé : on ne prolonge jamais en silence."""
    if not (rapports[0] <= r <= rapports[-1] and lambdas[0] <= lam <= lambdas[-1]):
        raise ValueError("hors de l'abaque : d/W de 0,1 à 0,8 et lambda de 0 à 1")
    return float(lecture([[r, lam]])[0])

print(refus)
print(f"d/W = 0,9 : exact {kt_090:.4f} ; prolongement linéaire {kt_prolonge:.4f} ; valeur de l'extrémité {kt_borne:.4f}")
assert refus.startswith("ValueError"), "RegularGridInterpolator refuse l'extrapolation par défaut"
assert abs(kt_borne - kt_noeuds[-1]) < 1e-12, "np.interp reste à la valeur du dernier nœud"
assert (
    abs(kt_prolonge - kt_090) < 0.02 and abs(kt_borne - kt_090) < 0.02
), "près du dernier nœud, les deux restent proches"
assert abs(kt_abaque(0.35, 0.5) - kt_2d) < 1e-12

## Exercice 14.3 : Abaque à deux entrées, et extrapolation refusée (défi, 25 min)

Avec la table du texte, lisez par la grille linéaire $K_t$ pour $d/W = 0{,}35$ et $\lambda = 0{,}5$, et comparez à la formule multipliée par $(3 - \lambda)/3$. Prolongez ensuite la table à $d/W = 0{,}9$ (avec $\lambda = 0$) par `fill_value=None` et chiffrez l'erreur. Écrivez enfin une fonction `kt_abaque(r, lam)` qui lit la grille et lève une `ValueError` hors de l'abaque, et vérifiez qu'elle refuse $d/W = 0{,}9$.

**Résultat attendu.** Grille de 1,8996, formule de 1,8955, grille cubique de 1,8955 ; à $d/W = 0{,}9$, formule de 2,0010 et prolongement de 1,9890, soit une erreur de 0,0120 ; la fonction refuse le point.

**Seconde voie.** La formule de Heywood multipliée par $(3 - \lambda)/3$ donne 1,8955 à $(0{,}35\,;\,0{,}5)$ et 2,0010 à $d/W = 0{,}9$.

**Indication.** La lecture est `lecture([[0.35, 0.5]])[0]`. Pour le prolongement, un second interpolateur avec `bounds_error=False, fill_value=None`. La fonction teste que $r$ et $\lambda$ sont entre les bornes de `rapports` et de `lambdas`, puis lève `ValueError("hors de l'abaque")` sinon. L'erreur de prolongement reste faible parce que $0{,}9$ est près du dernier nœud ; plus loin, elle croît.

Les contrôles de la cellule lisent les variables `erreur_prolongement`, `refuse` : gardez ces noms.

In [ ]:
# Votre code :
assert abs(kt_2d - 1.8996) < 5e-5, "Kt de la grille attendu : 1,8996"
assert abs(kt_2d_exact - 1.8955) < 5e-5, "Kt exact attendu : 1,8955"
assert refuse, "la fonction refuse d/W = 0,9"
assert (
    abs(erreur_prolongement) < 0.02
), "le prolongement de la pente reste proche près du dernier nœud"
print("Exercice 14.3 : contrôles passés")

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F14-1 : Interpolation sur grille (`scipy.interpolate.RegularGridInterpolator`)

In [ ]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator
T = np.linspace(20.0, 80.0, 4)           # températures, °C
v = np.linspace(1.0, 5.0, 5)             # vitesses, m/s
h = 10 + 0.5 * T[:, None] + 12 * v[None, :] + 0.1 * T[:, None] * v[None, :]
abaque = RegularGridInterpolator((T, v), h)
h_lu = abaque([[50.0, 2.5]])[0]          # lecture à 50 °C, 2,5 m/s
h_vrai = 10 + 0.5 * 50.0 + 12 * 2.5 + 0.1 * 50.0 * 2.5
print(f"Lu : {h_lu:.1f} W/(m²·K)".replace(".", ","))
print("Contrôle :", np.isclose(h_lu, h_vrai))